# DDColor

In [ ]:
import torch
import torch.nn as nn
from torch.nn import MultiheadAttention

### Backbone
**ConvNeXt**

**input:**

(N, 1, 256, 256)

**output:**

(N, 96, 64, 64)

(N, 192, 32, 32)

(N, 384, 16, 16)

(N, 768, 8, 8)

In [ ]:
class ConvNeXtBlock(nn.Module):
  def __init__(self, dim):
    super(ConvNeXtBlock, self).__init__()
    self.dconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
    self.norm = nn.GroupNorm(1, dim, eps=1e-6)
    self.pwconv1 = nn.Linear(dim, 4 * dim)
    self.gelu = nn.GELU()
    self.pwconv2 = nn.Linear(4 * dim, dim)

  def forward(self, x):
    shortcut = x
    x = self.dconv(x)
    x = self.norm(x)
    x = x.permute(0, 2, 3, 1)
    x = self.pwconv1(x)
    x = self.gelu(x)
    x = self.pwconv2(x)
    x = x.permute(0, 3, 1, 2)
    x = x + shortcut
    return x

In [ ]:
class ConvNeXt(nn.Module):
  def __init__(self, in_channels=1):
    super(ConvNeXt, self).__init__()

    self.downsample1 = nn.Sequential(nn.Conv2d(in_channels, 96, kernel_size=4, stride=4), nn.GroupNorm(1, 96, eps=1e-6))
    self.stage1 = nn.Sequential(*[ConvNeXtBlock(96) for _ in range(3)])

    self.downsample2 = nn.Sequential(nn.Conv2d(96, 192, kernel_size=2, stride=2), nn.GroupNorm(1, 192, eps=1e-6))
    self.stage2 = nn.Sequential(*[ConvNeXtBlock(192) for _ in range(3)])

    self.downsample3 = nn.Sequential(nn.Conv2d(192, 384, kernel_size=2, stride=2), nn.GroupNorm(1, 384, eps=1e-6))
    self.stage3 = nn.Sequential(*[ConvNeXtBlock(384) for _ in range(9)])

    self.downsample4 = nn.Sequential(nn.Conv2d(384, 768, kernel_size=2, stride=2), nn.GroupNorm(1, 768, eps=1e-6))
    self.stage4 = nn.Sequential(*[ConvNeXtBlock(768) for _ in range(3)])

  def forward(self, x):
    # Input: (N, 1, H, W)
    x = self.downsample1(x)
    # (N, 96, H/4, W/4)
    x1 = self.stage1(x)

    x = self.downsample2(x1)
    # (N, 192, H/8, W/8)
    x2 = self.stage2(x)

    x = self.downsample3(x2)
    # (N, 384, H/16, W/16)
    x3 = self.stage3(x)

    x = self.downsample4(x3)
    # (N, 768, H/32, W/32)
    x4 = self.stage4(x)

    return x1, x2, x3, x4

### Fusion Module

In [ ]:
class Fusion(nn.Module):
  def __init__(self):
    super(Fusion, self).__init__()
    self.conv = nn.Conv2d(in_channels=100, out_channels=2, kernel_size=1, stride=1)
    self.activation = nn.Tanh()

  def forward(self, E_c, E_i):
    '''
    inputs:
      E_c: output of the color decoder: (N, K, C)
      E_i: output of the pixel decoder: (N, C, H, W)
    '''
    N, K, C = E_c.shape
    _, _, H, W = E_i.shape

    # do dot product between E_c and E_i resultiung in (N, K, H, W)
    E_i_reshaped = E_i.view(N, C, -1)
    x = torch.bmm(E_c, E_i_reshaped)
    x = x.view(N, K, H, W)

    # apply convolution to change num of channels to 2 from K(100)
    x = self.conv(x)
    x = self.activation(x)

    return x

### CDB(Color Decoder Block)

In [ ]:
class CDBLayer(nn.Module):
  def __init__(self, C_l, C, num_heads):
    super(CDBLayer, self).__init__()
    self.conv_K = nn.Conv2d(C_l, C, kernel_size=1)
    self.conv_V = nn.Conv2d(C_l, C, kernel_size=1)

    self.conv_ff_1 = nn.Conv2d(C, 2048, kernel_size=1)
    self.conv_ff_2 = nn.Conv2d(2048, C, kernel_size=1)

    self.cross_attention = MultiheadAttention(C, num_heads, batch_first=True)
    self.self_attention = MultiheadAttention(C, num_heads, batch_first=True)

    self.layer_norm1 = nn.LayerNorm(C)
    self.layer_norm2 = nn.LayerNorm(C)
    self.layer_norm3 = nn.LayerNorm(C)

  def forward(self, Q, K, V):
    '''
    inputs:
      K: (N, C_l, H_l, W_l)
      V: (N, C_l, H_l, W_l)
      Q: (K, C)
    '''
    # first apply 1X1 conv to chnage number of channels of K and V to C(256) from C_l
    K = self.conv_K(K)
    V = self.conv_V(V)

    # now K and V has shape of (N, 256, H_l, C_l)
    K = K.flatten(2).transpose(1, 2)
    V = V.flatten(2).transpose(1, 2)

    # now K and V has shape of (N, H_l*C_l, C)

    # Cross-Attention
    Q_prime = Q # (N, K, C)
    Q, _ = self.cross_attention(Q, K, V)
    Q = self.layer_norm1(Q + Q_prime)

    # Self-Attention
    # (N, K, C)
    Q_prime = Q
    Q, _ = self.self_attention(Q, Q, Q)
    Q = self.layer_norm2(Q + Q_prime)

    # Feed-Forward Network
    # (N, K, C)
    Q_prime = Q
    Q = Q.transpose(1, 2).unsqueeze(-1)
    Q = self.conv_ff_1(Q)
    Q = self.conv_ff_2(Q)
    Q = Q.squeeze(-1).transpose(1, 2)
    Q = self.layer_norm3(Q + Q_prime)

    # (N, K, C)
    return Q

In [ ]:
class CDB(nn.Module):
  def __init__(self, d_model=256, num_queries=100, num_heads=8, num_groups=3, blocks_per_group=3):
    super(CDB, self).__init__()
    self.num_groups = num_groups
    self.blocks_per_group = blocks_per_group

    self.cdb_layers = nn.ModuleList([
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=256, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=256, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=512, C=d_model, num_heads=num_heads),
        CDBLayer(C_l=256, C=d_model, num_heads=num_heads)
    ])

    self.Q = nn.Parameter(torch.zeros(num_queries, d_model))

  def forward(self, KV1, KV2, KV3):
    batch_size = KV1.size()[0]
    Q = self.Q.unsqueeze(0).expand(batch_size, -1, -1)  # (N, K, d_model)
    for i, layer in enumerate(self.cdb_layers):
      if i % 3 == 0:
        KV = KV1
      elif i % 3 == 1:
        KV = KV2
      else:
        KV = KV3

      Q = layer(Q, KV, KV)

    return Q

### Pixel Deocder

**inputs:**

(N, 768, H/32, W/32)

**shortcut outputs:**

(N 384, H/16, W/16)

(N, 192, H/8, W/8)

(N, 96, H/4, W/4)

(N, 1, H,W)

**output:**

(N, C=256, H,W)

In [ ]:
class PixelDecoder(nn.Module):
  def __init__(self):
    super(PixelDecoder, self).__init__()
    self.pixel_shuffle_2 = nn.PixelShuffle(upscale_factor=2)
    self.pixel_shuffle_4 = nn.PixelShuffle(upscale_factor=4)
    self.conv1 = nn.Conv2d(in_channels=576, out_channels=512, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(in_channels=320, out_channels=512, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(in_channels=224, out_channels=256, kernel_size=3, padding=1)
    self.conv4 = nn.Conv2d(in_channels=17, out_channels=256, kernel_size=3, padding=1)

  def forward(self, x, shortcut1, shortcut2, shortcut3, shortcut4):
    # x: (N, 768, 8, 8)
    # shortcut1: (N, 384, 16, 16)
    # shortcut2: (N, 192, 32, 32)
    # shortcut3: (N, 96, 64, 64)
    # shortcut4: (N, 1, 256, 256)

    x = self.pixel_shuffle_2(x) # (N, 192, 16, 16)
    x = torch.concatenate([x, shortcut1], axis=1) # (N, 576, 16, 16)
    x = self.conv1(x) # (N, 512, 16, 16)
    f_out1 = x

    x = self.pixel_shuffle_2(x) # (N, 128, 32, 32)
    x = torch.concatenate([x, shortcut2], axis=1) # (N, 320, 32, 32)
    x = self.conv2(x) # x: (N, 512, 32, 32)
    f_out2 = x

    x = self.pixel_shuffle_2(x) # (N, 128, 64, 64)
    x = torch.concatenate([x, shortcut3], axis=1) # (N, 224, 64, 64)
    x = self.conv3(x) # (N, 256, 64, 64)
    f_out3 = x

    x = self.pixel_shuffle_4(x) # (N, 16, 256, 256)
    x = torch.concatenate([x, shortcut4], axis=1) # (N, 17, 256, 256)
    E_i = self.conv4(x)  # (N, 256, 256, 256)

    return f_out1, f_out2, f_out3, E_i

### Putting it all together

In [ ]:
class DDColorModel(nn.Module):
  def __init__(self):
    super(DDColorModel, self).__init__()
    self.backbone = ConvNeXt()
    self.pixel_decoder = PixelDecoder()
    self.fusion = Fusion()
    self.color_decoder = CDB(num_queries=100, d_model=256, num_heads=8)

  def forward(self, input_image):
    out1, out2,out3, out4 = self.backbone(input_image)
    f_out1, f_out2, f_out3, E_i = self.pixel_decoder(out4, out3, out2, out1, input_image)
    E_c = self.color_decoder(f_out1, f_out2, f_out3)
    real_final_output = self.fusion(E_c, E_i)
    return real_final_output